# Steps
- Extract: fetch data from API using `requests`
- Transform: read raw data from memory bytes using BytesIO and parse CSV or JSON formats
- Aggregate: group data, calculate sums/averages, filter rows
- Load: save the final output to a new file, DB, or return it as bytes

## Script version

In [9]:
import csv
import io
import requests
from collections import defaultdict

In [ ]:
# 1. Extract
url = "https://fsu.edu"
response = requests.get(url)
response.raise_for_status()

In [ ]:
# 2. Transform - convert response content to a file-like text stream
    # we don't want to do response.json or response.text because they load the entire payload into your system's RAM all at once and could cause the system to crash if it runs out of memory
    # using BytesIO /TextIOWrapper with streaming instead writes the data to an in-memory binary byte stream
        # allows us to process data line-by-line
file_stream = io.BytesIO(response.content) 
text_stream = io.TextIOWrapper(file_stream, encoding="utf-8")
reader = csv.DictReader(text_stream)

# Transform - streaming version

In [ ]:
# TRANSFORM - STREAMING VERSION**
url = "https://fsu.edu"

# keep connection open without downloading everything into RAM
response = requests.get(url, stream=True)
response.raise_for_status()

# iter_lines() yields chunks of bytes as they arrive
# TextIOWrapper not needed here because decode_unicode handles bytes --> str
lines = response.iter_lines(decode_unicode=True)

# DictReader processes the stream row-by-row lazily
reader = csv.DictReader(lines)


In [ ]:
# Summary of ingestion methods 

# Small JSON responses: response.json()
# Small CSV responses: io.StringIO(response.text) --> csv.DictReader
# Massive files: requests.get(stream=True) and iter_lines()


# TextIOWrapper: use when reading or writing actual text files or byte streams on disk
    # decoding bytes: wrap a binary stream to decode raw bytes into readable string characters
    # large data streaming: process large datasets piece-by-piece from disk to avoid loading everything into computer memory at once
# StringIO: use when working with pure in-memory text buffers
    # in-memory strings: treat simple Python string variables as a file-like stream
    # ETL string manupulation: you collect generated CSV or JSON text data in memory

# io.BytesIO: stores binary response data in RAM so you don't need to save the physical file to disk

In [ ]:
# 3. Aggregate
total_rows = 0
state_counts = defaultdict(int)

for row in reader:
    total_rows += 1
    state = row.get("State", "").strip()
    if state:
        state_counts[state] += 1

In [8]:
# 4. Load
print(f"Total rows processed: {total_rows}")
print("State aggregations:")
for state, count in state_counts.items():
    print(f"- {state}: {count}")

Total rows processed: 0
State aggregations:


In [ ]:
# -----------------------------------------

## In-Memory ETL Blueprint

In [ ]:
# handles both CSV and JSON streams from an API entirely within memory
import csv
import io
import json
import requests

In [ ]:
def run_etl_pipeline(url, data_format):
    # 1. EXTRACT
    response = requests.get(url)
    response.raise_for_status()

    # read raw binary data into memory; since it avoids writing to local disk, it optimizes performance
    raw_bytes = io.BytesIO(response.content) # response.content yields bytes, which then need to be wrapped with TextIOWrapper to convert to string before being passed to json.load()


    # 2. TRANSFORM
    parsed_data = []
    text_stream = io.TextIOWrapper(raw_bytes, encoding="utf-8")

    if data_format.lower() == "json":
        parsed_data = json.load(text_stream)

    elif data_format.lower() == "csv":
        reader = csv.DictReader(text_stream)

        for row in reader:
            cleaned_row = { k.strip().lower(): v.strip() for k, v in row.items() }
            parsed_data.append(cleaned_row)


    transformed_data = []
    for record in parsed_data:
        try:
            category = record.get("category", "Unknown").title()
            price = float(record.get("price", 0.0))
            quantity = int(record.get("quantity", 0))

            # filter out invalid records
            if price <= 0 or quantity <= 0:
                continue

            total_revenue = price * quantity

            transformed_data.append({"category": category, "total_revenue": total_revenue})
        except (ValueError, TypeError) as e:
            print(f"Exception occurred: {e}")
            continue

    # 3. AGGREGATE
    aggregation_results = {}
    for item in transformed_data:
        cat = item["category"]
        rev = item["total_revenue"]
        aggregation_results[cat] = aggregation_results.get(cat, 0.0) + rev


    # 4. LOAD
    return aggregation_results

In [ ]:
# -----------------------------------------

## OOP Version - 1

In [ ]:
import csv
import io
import json
import requests

In [ ]:
# Single responsibility: each step (extract, transform, aggregate) has a single responsibility/separation of concerns
# Encapsulation + state: the raw data state (self.raw_data) and transformed state (self.transformed_data) are isolated within the instance, preventing side-effects

class BaseETLPipeline:
    def __init__(self, url, data_format="json"):
        self.url = url
        self.data_format = data_format.lower()
        self.raw_data = []
        self.transformed_data = []

    # fetches binary data from the network into an in-memory byte buffer
    def extract(self):
        response = requests.get(self.url)
        response.raise_for_status()
        return io.BytesIO(response.content)

    def _clean_row(self, row):
        return { k.strip().lower(): v.strip() for k, v in row.items() }

    # parses raw formats, handles missing/corrupt rows, and applies business logic
    def transform(self, buffer):
        text_stream = io.TextIOWrapper(buffer, encoding="utf-8")

        # 1. parse raw format
        if self.data_format == "json":
            self.raw_data = json.load(text_stream)
        elif self.data_format == "csv":
            reader = csv.DictReader(text_stream)
            self.raw_data = [self._clean_row(row) for row in reader]

        # 2. data enrichment and filtering 
        for record in self.raw_data:
            try:
                # handle missing keys gracefully
                if "price" not in record or "quantity" not in record:
                    continue

                category = record.get("category", "Unknown").strip().title()
                price = float(record.get("price"))
                quantity = int(record.get("quantity"))

                # filter out invalid/anomaly business data
                if price <= 0 or quantity <= 0:
                    continue

                self.transformed_data.append(
                    {
                        "category": category,
                        "total_revenue": price*quantity
                    }
                )

            except (ValueError, TypeError) as e:
                continue

    def aggregate(self):
        results = {}
        for item in self.transformed_data:
            cat = item["category"]
            rev = item["total_revenue"]
            results[cat] = results.get(cat, 0.0) + rev
        return results

    def run(self):
        data_buffer = self.extract()
        self.transform(data_buffer)
        return self.aggregate()


In [ ]:
# -----------------------------------------

## OOP - Streaming ETL Pipeline Pattern

In [ ]:
import csv
import io
import json
import requests
from typing import Generator, Dict, Any

In [ ]:
# memory-efficient pipeline that streams and processes data line-by-line
class StreamingETLPipeline:
    def __init__(self, url, data_format="json"):
        self.url = url
        self.data_format = data_format.lower()

    def extract_stream(self):
        # streams raw content from the network line-by-line
        # using stream=True prevents loading the entire payload into RAM at once

        # stream=True keeps the network connection open to download data in chunks
        with requests.get(self.url, stream=True) as response:
            response.raise_for_status()

            # iter_lines yields individual lines as decoded strings (bytes --> str)
            # bypasses need for manual BytesIO text wrappers
            for line in response.iter_lines(decode_unicode=True):
                if line:
                    yield line # Generator

    def _clean_row(self, row):
        return {
            k.strip().lower(): v.strip() for k, v in row.items()
        }

    def transform_stream(self, line_generator):
        # parses text streams lazily and yields cleaned, schema-validated dictionaries

        if self.data_format == "csv":
            # DictReader accepts any generator/iterator that yields lines of text
            reader = csv.DictReader(line_generator)
            for row in reader:
                cleaned_row = self._clean_row(row)
                try:
                    category = cleaned_row.get("category", "Unknown").strip().title()
                    price = float(cleaned_row.get("price"))
                    quantity = int(cleaned_row.get("quantity"))

                    if price <= 0 or quantity <= 0:
                        continue

                    yield {
                        "category": category,
                        "total_revenue": price*quantity
                    }
                except (KeyError, ValueError, TypeError):
                    continue

        elif self.data_format == "json":
            # streaming JSON requires lines to be JSONL
            for line in line_generator:
                try:
                    record = json.loads(line)
                    cleaned_record = self._clean_row(record)

                    category = cleaned_record.get("category", "Unknown").strip().title()
                    price = float(cleaned_record.get("price"))
                    quantity = int(cleaned_record.get("quantity"))

                    if price <= 0 or quantity <= 0:
                        continue

                    yield {
                        "category": category,
                        "total_revenue": price*quantity
                    }

                except (json.JSONDecodeError, KeyError, ValueError, TypeError):
                    continue

    def aggregate(self, transformed_generator):
        results = {}
        for item in transformed_generator:
            cat = item.get("category")
            rev = item.get("total_revenue")
            results[cat] = results.get(cat, 0.0) + rev
        return results

    def run(self):
        # extract_stream + transform_stream do not process data/occupy massive memory immediately -- they return generator objects
            # data is only pulled through the pipeline during `for item in transformed_generator` within aggregate()
        raw_lines = self.extract_stream()
        transformed_records = self.transform_stream(raw_lines)
        return self.aggregate(transformed_records)


if __name == "__main__":
    pipeline = StreamingETLPipeline(
        url="https://example.com",
        data_format="csv"
    )

results = pipeline.run()

In [ ]:
# -----------------------------------------

## Log parsing example

- given a list of raw transaction logs as strings
- each log contains a comma-separated format: timestamp, transaction_id, user_id, action, amount
- write a function to:
    - filter out any actions that aren't 'PURCHASE'
    - deduplicate by transaction_id
    - calculate the total amount spent per user
    - return a dictionary of results
    - error handling

In [13]:
import logging
from collections import defaultdict

logging.basicConfig(level=logging.ERROR)

In [ ]:
def process_user_spending(raw_logs):

    # 1. initialize lookup trackers (O(1))
    seen_transactions = set()
    user_spending = {}

    # right now, raw_logs is passed as a list, held in memory
        # if this file was significantly larger, I would change the input to a file path or buffer and stream it line-by-line using a python generator (with open(file) as f: for line in f: yield)
    for line in raw_logs:
        cleaned_line = line.strip()
        if not cleaned_line:
            continue

        try:
            # 2. parse and tokenize row
            parts = [p.strip() for p in cleaned_line.split(",")]
            if len(parts) != 5:
                raise ValueError("Malformed row layout")
            timestamp, tx_id, user_id, action, amount_str = parts

            # 3. apply business logic + filtering
            if action != "PURCHASE":
                continue

            if tx_id in seen_transactions:
                continue

            # 4. cast and validate data types
            amount = float(amount_str)

            # 5. commit states and aggregate data
            seen_transactions.add(tx_id)
            user_spending[user_id] = user_spending.get(user_id, 0.0) + amount

        except (ValueError, IndexError) as e:
            # log corrupt data without crashing pipeline
            logging.error(f"Skipping corrupted row: {e}")
            continue

    return user_spending

In [ ]:
if __name__ == "__main__": 
    mock_logs = [
        "2026-01-01 10:00, tx01, user_A, PURCHASE, 150.50",
        "2026-01-01 10:01, tx02, user_B, VIEW, 0.00",  # Should be filtered out 
        "2026-01-01 10:02, tx01, user_A, PURCHASE, 150.50", # Should be deduplicated 
        "2026-01-01 10:03, tx03, user_A, PURCHASE, 50.00",  # New purchase for user_A 
        "2026-01-01 10:04, tx04, user_B, PURCHASE, 300.00", # New purchase for user_B 
        "BAD_ROW_DATA, missing_fields",  # Corrupt row, should log and skip 
    ]

    result = process_user_spending(mock_logs)
    print("Final aggregation: ", result)

ERROR:root:Skipping corrupted row: Malformed row layout


Final aggregation:  {'user_A': 200.5, 'user_B': 300.0}


In [ ]:
# -----------------------------------------

## Log parsing -- streaming

In [20]:
import io
import logging

logging.basicConfig(level=logging.ERROR)

In [24]:
def stream_log_lines(file_path):
    # open file and yield rows one-by-one, replacing loading an entire massive list into memory
    # returns a generator object instantly without running the code inside - lazy evaluation
    with open(file_path, mode="r", encoding="utf-8") as file:
        for line in file:
            yield line


# right now, we're storing all transaction ID's in memory in seen_transactions
    # if we had 10 billion transactions, this would cause an out-of-memory crash
    # we could swap this set for something like Redis to keep the script's memory entirely flat
def process_streaming_spending(log_stream):
    # transform + aggregate
    # time complexity: O(n)
    seen_transactions = set()
    user_spending = {}

    for line in log_stream:
        cleaned_line = line.strip()
        if not cleaned_line:
            continue

        try:
            parts = [p.strip() for p in cleaned_line.split(",")]
            print(parts)
            if len(parts) != 5:
                length = len(parts)
                raise ValueError(f"Malformed row layout, len(parts): {length}")

            timestamp, tx_id, user_id, action, amount_str = parts

            if action != "PURCHASE" or tx_id in seen_transactions:
                continue

            amount = float(amount_str)
            seen_transactions.add(tx_id)
            user_spending[user_id] = user_spending.get(user_id, 0.0) + amount

        except (ValueError, IndexError) as e:
            logging.error(f"Skipping corrupted row: [{cleaned_line}], error: {e}")
            continue

    print(user_spending)
    return user_spending


if __name__ == "__main__":
    mock_file_buffer = io.StringIO("""
        2026-01-01 10:00, tx01, user_A, PURCHASE, 150.50
        2026-01-01 10:01, tx02, user_B, VIEW, 0.00
        2026-01-01 10:02, tx01, user_A, PURCHASE, 150.50
        2026-01-01 10:03, tx03, user_A, PURCHASE, 50.00
        2026-01-01 10:04, tx04, user_B, PURCHASE, 300.00
        BAD_ROW_DATA, missing_fields
    """)

    generator_input = (line for line in mock_file_buffer) # create generator stream via Generator Expression
    result = process_streaming_spending(generator_input) # process stream on the fly


# Brackets vs Parentheses
    # 1. Brackets - List Comprehension
        # this runs immediately
        # creates all elements and holds them in RAM

    # 2. Parentheses - Generator Expression
        # does NOT run immediately
        # creates a lazy iterator object

ERROR:root:Skipping corrupted row: [BAD_ROW_DATA, missing_fields], error: Malformed row layout, len(parts): 2


['2026-01-01 10:00', 'tx01', 'user_A', 'PURCHASE', '150.50']
['2026-01-01 10:01', 'tx02', 'user_B', 'VIEW', '0.00']
['2026-01-01 10:02', 'tx01', 'user_A', 'PURCHASE', '150.50']
['2026-01-01 10:03', 'tx03', 'user_A', 'PURCHASE', '50.00']
['2026-01-01 10:04', 'tx04', 'user_B', 'PURCHASE', '300.00']
['BAD_ROW_DATA', 'missing_fields']
{'user_A': 200.5, 'user_B': 300.0}


In [ ]:
# -----------------------------------------

## Top K most active zip codes

- you are streaming a live feed of property view lows
- each log is a simple string format: property_id, zip_code
- write a memory-efficient function that processes this stream and returns the top K most active zipcodes

In [25]:
import io
import logging
from collections import Counter

logging.basicConfig(level=logging.ERROR)

In [ ]:
def get_top_k_zipcodes(log_stream, k):
    zip_counts = Counter() # Counter keeps data processing down to a single pass (O(n) time). We stream directly into it, so we never hold the whole file list in memory
        # if there are too many zipcodes to hold in memory, we could use distributed streaming instead

    for line in log_stream:
        cleaned_line = line.strip()
        if not cleaned_line:
            continue

        try:
            parts = [p.strip() for p in cleaned_line.split(",")]
            if len(parts) != 2:
                raise ValueError("Invalid row format")
            
            property_id, zip_code = parts

            if not zip_code.isdigit() or len(zip_code) != 5:
                raise ValueError(f"Malformed zipcode: {zip_code}")

            zip_counts[zip_code] += 1

        except ValueError as e:
            logging.error(f"Skipping row error: {e}")
            continue 

    # the other option would be calling sorted(), but that's significantly slower than .most_common() which uses a more efficient min heap
    return zip_counts.most_common(k)
    

if __name__ == "__main__":
    mock_stream_buffer = io.StringIO("""
        prop_1,98101
        prop_2,98101
        prop_3,98111
        prop_4,4
        prop_5,77777
        prop_6,BAD
        prop_7,94103
        prop_8,98101
        prop_9,94103
        prop_10,98111
    """)

    stream = (line for line in mock_stream_buffer)

    top_4_zips = get_top_k_zipcodes(stream, k=4)
    print(top_4_zips)


ERROR:root:Skipping row error: Malformed zipcode: 4
ERROR:root:Skipping row error: Malformed zipcode: BAD


[('98101', 3), ('98111', 2), ('94103', 2), ('77777', 1)]


# Reading practice

In [ ]:
# StringIO
    # ** Text in memory **
    # input type: string
    # best for: mocking local text files or CSV

import io
mock_file_data = "id,price\n1,5000\n2,6000"

file_obj = io.StringIO(mock_file_data)
for line in file_obj:
    print(line.strip())


id,price
1,5000
2,6000


In [39]:
# BytesIO
    # ** Bytes in memory **
    # input type: bytes
    # best for: mocking downloaded zip files, images, compressed parquet

import io
binary_data = b"Some raw compressed bytes from an API or S3"
binary_stream = io.BytesIO(binary_data)

data = binary_stream.read()

In [40]:
# TextIOWrapper
    # ** Translator: Bytes --> Text **
    # input type: bytes
    # best for: converting stream of raw bytes into a stream of readable text strings

import io

# wrap binary stream and translate it to text on-the-fly
text_stream = io.TextIOWrapper(binary_stream, encoding="utf-8")

for line in text_stream:
    print(line)

In [41]:
# iter_lines()
    # requests library
    # use when you hit an API endpoint that returns a massive dataset
    # allows you to consume it line-by-line over the network without downloading the entire payload into memory first

import requests

# stream=True keeps the network connection open without downloading everything
response = requests.get("https://redfin.com", stream=True)

for line in response.iter_lines(decode_unicode=True):
    if line:
        print(line)

<!doctype html><html><head><meta charset="utf-8"><title>Are You a Robot? | Redfin</title><style>body {font-family: "Helvetica Neue", Helvetica, Arial, sans-serif;margin: 0;text-align: left;font-size: 16px;color: #333;}#header {min-width: 300px;height: 60px;width: 100%;background-color: #fff;}#header .logo {padding: 1rem 0;display: inline-block;}#header.WhiteHeaderContainer .Header {box-shadow: 0 0 1px rgba(0, 0, 0, .3);padding: 0 10px;height: 60px;}#main {max-width: 100%;width: 650px;margin: 0 auto;text-align: center;}#sub {text-align: left;background-color: #e2e2e2;padding: 0.1em 1em;border-radius: 0.5em;}h2 {font-size: 28px;font-weight: 300;}p {font-weight: 200;}li:not(:last-child) {margin-bottom: 0.5em;}textarea {width: 100%;height: 15em;overflow: auto;}</style></head><body><div id="header" class="WhiteHeaderContainer mobile-enabled-header"><header id="header_content" class="Header" style="min-height: 45px;"><div class="logo"><img itemprop="logo" class="" style="height:32px;" alt="R

# Bytes + base64 - CSV

In [44]:
import base64
import io
import csv

In [45]:
# base64.b64decode() --> translator
    # takes text that looks like scrambled ASCII characters and decodes it back into raw binary bytes
    # .decode() --> converts binary bytes into readable Python string text
# io.BytesIO() --> parking space in RAM
    # takes those raw binary bytes and wraps them in a file-like interface so other libraries can read them as if they were an actual file
    # avoids disk I/O writing a temporary file

# raw input: base64 text string from JSON payload representing a CSV file
api_response_string = "aWQscHJpY2UKMSw1MDAwMDAKMiw2MDAwMDA=" 

# 1. decode base64 string into raw binary bytes (translate text back into binary data)
raw_binary_bytes = base64.b64decode(api_response_string)
print(raw_binary_bytes)

# 2. turns those bytes back into a memory stream/mock file object in RAM
bytes_file_handle = io.BytesIO(raw_binary_bytes)

# 3. wrap memory stream to translate binary into text strings
text_stream = io.TextIOWrapper(bytes_file_handle, encoding='utf-8')

# 4. pass the stream to csv.DictReader
reader = csv.DictReader(text_stream)

for row in reader:
    print(row)

b'id,price\n1,500000\n2,600000'
{'id': '1', 'price': '500000'}
{'id': '2', 'price': '600000'}


# Bytes + base64 - Direct Line-Streaming

In [ ]:
import base64
import io

api_response_string = "aWQscHJpY2UKMSw1MDAwMDAKMiw2MDAwMDA=" 

# decode to binary bytes and stream via BytesIO
raw_binary_bytes = base64.b64decode(api_response_string)
bytes_file_handle = io.BytesIO(raw_binary_bytes)

# read stream row-by-row, decoding each line individually
for binary_line in bytes_file_handle:
    # convert individual binary row back to a Python string text
    text_line = binary_line.decode('utf-8').strip()
    print(text_line)

# file handler - pointer to a file so you don't read it all into memory
    # keep a lightweight file handle reference so we can stream data lazily through an iterator, rather than calling the read method that pulls the entire payload into memory at once

id,price
1,500000
2,600000


In [51]:
keys = ["name", "age", "gender"]
values = ["Bob", 42, "M"]

dic = dict(zip(keys, values))

print(dic)
print(list(dic))



{'name': 'Bob', 'age': 42, 'gender': 'M'}
['name', 'age', 'gender']


In [53]:
names = ["Alice", "Bob", "Caleb"]
scores = [92, 85, 78]

for rec in zip(names, scores):
    print(rec)

('Alice', 92)
('Bob', 85)
('Caleb', 78)


# Bytes + base64 - Direct Line-Streaming